# Retention YTD Training Objective Handoff v1

Runtime card: `runtime/model_selection/card/retention_ytd_training_objective_handoff_v1`

Purpose: translate the current retained-model requirement into a concrete next training/checkpoint-selection contract. This card consumes the latest retention aggregate, SOXLBacktest qlib reference, and official warm-start state-conditioned evidence. It does **not** promote an actor and does **not** treat SOXLBacktest as runnable inference; it defines the hard gates and stop rules the next model branch must satisfy.

## Governance

- **Baselines**: canonical production composer `v19reg_v22_budget_usage_bull_guard_v1`; reference baseline `soxlbacktest_policy_reimplementation_v1`; direct diagnostic baselines `state_conditioned_multi_objective_bc_screen_v1`, `sequence_trend_joint_warm_start_official_eval_v1`, and `soxlbacktest_qlib_walkforward_baseline_v1`.
- **Role**: replace candidate handoff / training-contract diagnostic, not promotion.
- **Target windows**: `wf_2020`, `wf_2022`, `wf_2023_2024`, and YTD through safe replay end.
- **Changed layer**: model selection, training target, checkpoint scoring, reward constraints.
- **Hypothesis**: the next branch must learn state-conditioned participation only when risk state allows it; post-hoc floors and path-level threshold overlays have already failed the joint return/drawdown tradeoff.
- **Acceptance**: complete official replay evidence, worst WF MaxDD `<20%`, YTD return `>=213.143462%`, YTD return `>=132.077978%`, and no metric-source mismatch.
- **Failure mode**: YTD participation is recovered by increasing exposure but `wf_2022` or another WF window breaches the drawdown gate.

In [1]:

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists():
            return path
    raise RuntimeError(f'could not locate repo root from {start}')


ROOT = find_repo_root(Path.cwd())
OUT = ROOT / 'runtime/model_selection/card/retention_ytd_training_objective_handoff_v1'
OUT.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'retention_summary': ROOT / 'runtime/model_selection/card/retention_gap_aggregate_v1/summary.json',
    'retention_gate': ROOT / 'runtime/model_selection/card/retention_gap_aggregate_v1/candidate_gate_summary.csv',
    'retention_mechanism': ROOT / 'runtime/model_selection/card/retention_gap_aggregate_v1/mechanism_gap_summary.csv',
    'soxlbacktest_gate': ROOT / 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_gate_summary.csv',
    'multi_obj_selection': ROOT / 'runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/selection.json',
    'multi_obj_summary': ROOT / 'runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/summary.json',
    'multi_obj_candidate_summary': ROOT / 'runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/candidate_summary.csv',
    'multi_obj_official_metrics': ROOT / 'runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/warm_start_official_metrics.csv',
    'multi_obj_drawdown_intervals': ROOT / 'runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/drawdown_interval_diagnostic.csv',
    'sequence_summary': ROOT / 'runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1/summary.json',
    'sequence_selection': ROOT / 'runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1/selection.json',
    'dashboard_freshness': ROOT / 'dashboard_data/model_replay_freshness.json',
}


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open(encoding='utf-8') as fh:
        return json.load(fh)


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def clean_json(value):
    if isinstance(value, dict):
        return {str(key): clean_json(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clean_json(item) for item in value]
    if isinstance(value, tuple):
        return [clean_json(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


retention_summary = read_json(SOURCES['retention_summary'])
retention_gate = read_csv(SOURCES['retention_gate'])
retention_mechanism = read_csv(SOURCES['retention_mechanism'])
soxlbacktest_gate = read_csv(SOURCES['soxlbacktest_gate'])
multi_obj_selection = read_json(SOURCES['multi_obj_selection'])
multi_obj_summary = read_json(SOURCES['multi_obj_summary'])
multi_obj_candidate_summary = read_csv(SOURCES['multi_obj_candidate_summary'])
multi_obj_official_metrics = read_csv(SOURCES['multi_obj_official_metrics'])
multi_obj_drawdown_intervals = read_csv(SOURCES['multi_obj_drawdown_intervals'])
sequence_summary = read_json(SOURCES['sequence_summary'])
sequence_selection = read_json(SOURCES['sequence_selection'])
dashboard_freshness = read_json(SOURCES['dashboard_freshness'])

thresholds = retention_summary.get('target_thresholds', {})
TARGETS = {
    'required_wf_windows': thresholds.get('required_wf_windows', ['wf_2020', 'wf_2022', 'wf_2023_2024']),
    'wf_max_drawdown_hard_gate_pct': float(thresholds.get('max_walkforward_drawdown_pct', 20.0)),
    'soxlbacktest_reference_ytd_return_pct': float(thresholds.get('required_soxlbacktest_return_pct', 132.077978)),
    'current_best_ytd_return_pct': float(thresholds.get('current_best_return_pct', 224.361539)),
    'required_current_best_alignment_ytd_pct': float(thresholds.get('required_best_return_pct', 213.143462)),
    'official_internal_relative_return_floor_pp': 30.0,
    'official_internal_maxdd_margin_pct': 15.0,
    'safe_backtest_end': dashboard_freshness.get('safe_backtest_end'),
    'latest_data_date': dashboard_freshness.get('latest_data_date'),
}
TARGETS


{'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'wf_max_drawdown_hard_gate_pct': 20.0,
 'soxlbacktest_reference_ytd_return_pct': 132.077978,
 'current_best_ytd_return_pct': 224.361539,
 'required_current_best_alignment_ytd_pct': 213.143462,
 'official_internal_relative_return_floor_pp': 30.0,
 'official_internal_maxdd_margin_pct': 15.0,
 'safe_backtest_end': '2026-06-11',
 'latest_data_date': '2026-06-12'}

In [2]:

# Summarise source health and current best/closest rows.
source_audit = []
for label, path in SOURCES.items():
    exists = path.exists()
    rows = None
    if exists and path.suffix == '.csv':
        try:
            rows = int(len(pd.read_csv(path)))
        except Exception:
            rows = None
    source_audit.append({
        'source': label,
        'path': rel(path),
        'exists': bool(exists),
        'rows': rows,
    })
source_audit_df = pd.DataFrame(source_audit)

if retention_gate.empty:
    raise RuntimeError('retention aggregate candidate_gate_summary.csv is required')

ranked_retention = retention_gate.copy()
ranked_retention['objective_loss'] = pd.to_numeric(ranked_retention['objective_loss'], errors='coerce')
ranked_retention = ranked_retention.sort_values(['retention_gate_pass', 'objective_loss'], ascending=[False, True])
closest_retention = ranked_retention.iloc[0].to_dict()

wf_drawdown_pass = retention_gate[retention_gate.get('wf_drawdown_pass', False).astype(str).str.lower() == 'true'].copy()
if not wf_drawdown_pass.empty:
    wf_drawdown_pass['ytd_return_pct'] = pd.to_numeric(wf_drawdown_pass['ytd_return_pct'], errors='coerce')
    best_low_dd = wf_drawdown_pass.sort_values('ytd_return_pct', ascending=False).iloc[0].to_dict()
else:
    best_low_dd = {}

best_soxlbacktest = {}
if not soxlbacktest_gate.empty:
    sb = soxlbacktest_gate.copy()
    sb['retention_gate_pass'] = sb.get('retention_gate_pass', False).astype(str).str.lower() == 'true'
    sb['wf_drawdown_pass'] = sb.get('wf_drawdown_pass', False).astype(str).str.lower() == 'true'
    sb['ytd_return_pct'] = pd.to_numeric(sb.get('ytd_return_pct'), errors='coerce')
    best_soxlbacktest = sb.sort_values(['wf_drawdown_pass', 'ytd_return_pct'], ascending=[False, False]).iloc[0].to_dict()

source_audit_df


,source,path,exists,rows
0,retention_summary,runtime/model_selection/card/retention_gap_agg...,True,NaN
1,retention_gate,runtime/model_selection/card/retention_gap_agg...,True,61.0
2,retention_mechanism,runtime/model_selection/card/retention_gap_agg...,True,9.0
3,soxlbacktest_gate,runtime/model_selection/card/soxlbacktest_qlib...,True,6.0
4,multi_obj_selection,runtime/model_selection/card/state_conditioned...,True,NaN
5,multi_obj_summary,runtime/model_selection/card/state_conditioned...,True,NaN
6,multi_obj_candidate_summary,runtime/model_selection/card/state_conditioned...,True,3.0
7,multi_obj_official_metrics,runtime/model_selection/card/state_conditioned...,True,6.0
8,multi_obj_drawdown_intervals,runtime/model_selection/card/state_conditioned...,True,4.0
9,sequence_summary,runtime/model_selection/card/sequence_trend_jo...,True,NaN


In [3]:

# Convert existing evidence into a compact gap table for the next training contract.
gap_rows = []

def add_gap_row(source, candidate, role, metrics, notes):
    gap_rows.append({
        'source': source,
        'candidate': candidate,
        'role': role,
        'complete_wf': metrics.get('complete_wf'),
        'worst_wf_maxdd_pct': metrics.get('worst_wf_max_drawdown_pct', metrics.get('max_max_dd')),
        'wf_dd_gap_to_20_pct': max(0.0, float(metrics.get('worst_wf_max_drawdown_pct', metrics.get('max_max_dd', 0.0)) or 0.0) - TARGETS['wf_max_drawdown_hard_gate_pct']),
        'ytd_return_pct': metrics.get('ytd_return_pct'),
        'ytd_gap_to_current_best_alignment_pct': None if pd.isna(metrics.get('ytd_return_pct', np.nan)) else max(0.0, TARGETS['required_current_best_alignment_ytd_pct'] - float(metrics.get('ytd_return_pct'))),
        'ytd_gap_to_soxlbacktest_reference_pct': None if pd.isna(metrics.get('ytd_return_pct', np.nan)) else max(0.0, TARGETS['soxlbacktest_reference_ytd_return_pct'] - float(metrics.get('ytd_return_pct'))),
        'min_official_relative_return_pp': metrics.get('min_relative_return'),
        'official_maxdd_pct': metrics.get('max_max_dd'),
        'official_gate_loss': metrics.get('gate_loss'),
        'failures': metrics.get('failures'),
        'notes': notes,
    })

add_gap_row(
    'retention_gap_aggregate_v1',
    closest_retention.get('candidate_id', ''),
    'closest_retention_candidate',
    closest_retention,
    'Closest complete retained-model candidate still fails joint drawdown/YTD gate.',
)
if best_low_dd:
    add_gap_row(
        'retention_gap_aggregate_v1',
        best_low_dd.get('candidate_id', ''),
        'best_wf_drawdown_pass_by_ytd',
        best_low_dd,
        'Low-drawdown reference underparticipates versus both SOXLBacktest and current-best YTD floors.',
    )
if best_soxlbacktest:
    add_gap_row(
        'soxlbacktest_qlib_walkforward_baseline_v1',
        best_soxlbacktest.get('candidate', ''),
        best_soxlbacktest.get('role', 'reference_baseline'),
        best_soxlbacktest,
        'SOXLBacktest qlib buy/sell path is a reference baseline, not runnable inference.',
    )

if not multi_obj_candidate_summary.empty:
    mo = multi_obj_candidate_summary.copy()
    mo['gate_loss'] = pd.to_numeric(mo['gate_loss'], errors='coerce')
    best_mo = mo.sort_values(['full_gate_all', 'gate_loss'], ascending=[False, True]).iloc[0].to_dict()
    add_gap_row(
        'state_conditioned_multi_objective_bc_screen_v1',
        best_mo.get('candidate', ''),
        'official_warm_start_bc_candidate',
        best_mo,
        'State-conditioned BC recovered official relative return in some folds but breached drawdown margins.',
    )

if sequence_summary.get('candidate_summary'):
    seq = pd.DataFrame(sequence_summary['candidate_summary'])
    seq['gate_loss'] = pd.to_numeric(seq['gate_loss'], errors='coerce')
    best_seq = seq.sort_values(['full_gate_all', 'gate_loss'], ascending=[False, True]).iloc[0].to_dict()
    add_gap_row(
        'sequence_trend_joint_warm_start_official_eval_v1',
        best_seq.get('candidate', ''),
        'trend_sequence_reference',
        best_seq,
        'Earlier warm-start official reference shows risk control but misses recovery/YTD participation.',
    )

gap_table = pd.DataFrame(gap_rows)
gap_table


,source,candidate,role,complete_wf,worst_wf_maxdd_pct,wf_dd_gap_to_20_pct,ytd_return_pct,ytd_gap_to_current_best_alignment_pct,ytd_gap_to_soxlbacktest_reference_pct,min_official_relative_return_pp,official_maxdd_pct,official_gate_loss,failures,notes
0,retention_gap_aggregate_v1,existing_model_retention_gate::mode_conditione...,closest_retention_candidate,True,21.831720,1.831720,177.573569,35.569893,0.000000,NaN,NaN,NaN,worst_wf_max_drawdown_ge_20;ytd_return_below_c...,Closest complete retained-model candidate stil...
1,retention_gap_aggregate_v1,soxlbacktest_qlib_reference::wf_pass_ytd18_15_...,best_wf_drawdown_pass_by_ytd,True,18.514585,0.000000,18.308910,194.834552,113.769068,NaN,NaN,NaN,ytd_return_below_soxlbacktest_reference;ytd_re...,Low-drawdown reference underparticipates versu...
2,soxlbacktest_qlib_walkforward_baseline_v1,wf_pass_ytd18_15_15_02,wf_drawdown_pass_best_ytd_probe,None,18.514585,0.000000,18.308910,194.834552,113.769068,NaN,NaN,NaN,ytd_return_below_soxlbacktest_reference;ytd_re...,SOXLBacktest qlib buy/sell path is a reference...
3,state_conditioned_multi_objective_bc_screen_v1,multi_obj_floor45_cap20_riskfirst,official_warm_start_bc_candidate,None,30.626799,10.626799,NaN,NaN,NaN,30.262521,30.626799,167.488279,None,State-conditioned BC recovered official relati...
4,sequence_trend_joint_warm_start_official_eval_v1,trend_plain_contract,trend_sequence_reference,None,20.198957,0.198957,NaN,NaN,NaN,8.224563,20.198957,73.765009,None,Earlier warm-start official reference shows ri...


In [4]:

# Build explicit training/checkpoint objective recipe.
recipe_rows = [
    {
        'priority': 1,
        'term': 'official_replay_metric_source_required',
        'gate_type': 'data_contract',
        'threshold': 'all candidate metrics must come from official warm-start replay or dashboard replay freshness source',
        'implementation_target': 'checkpoint_selector_and_replay_pipeline',
        'reason': 'ordinary/cold split evidence has repeatedly gate-flipped under official replay',
    },
    {
        'priority': 2,
        'term': 'complete_wf_coverage_gate',
        'gate_type': 'hard_reject',
        'threshold': ','.join(TARGETS['required_wf_windows']),
        'implementation_target': 'candidate_manifest',
        'reason': 'retention aggregate shows missing WF evidence is a top failure count',
    },
    {
        'priority': 3,
        'term': 'walkforward_drawdown_hard_gate_20pct',
        'gate_type': 'hard_reject',
        'threshold': f"worst WF MaxDD < {TARGETS['wf_max_drawdown_hard_gate_pct']:.2f}%",
        'implementation_target': 'official_replay_checkpoint_screen',
        'reason': 'requested retained model must pass walk-forward drawdown below 20%',
    },
    {
        'priority': 4,
        'term': 'ytd_current_best_alignment_floor',
        'gate_type': 'hard_reject',
        'threshold': f"YTD return >= {TARGETS['required_current_best_alignment_ytd_pct']:.6f}% through safe_backtest_end={TARGETS['safe_backtest_end']}",
        'implementation_target': 'dashboard_replay_ytd_screen',
        'reason': 'low-drawdown paths underparticipate; current objective requires return alignment with existing best model',
    },
    {
        'priority': 5,
        'term': 'soxlbacktest_buysell_reference_floor',
        'gate_type': 'reference_floor',
        'threshold': f"YTD return >= {TARGETS['soxlbacktest_reference_ytd_return_pct']:.6f}%",
        'implementation_target': 'selection_score_reference_component',
        'reason': 'SOXLBacktest qlib buy/sell is a baseline floor, not a runnable profile',
    },
    {
        'priority': 6,
        'term': 'warm_start_drawdown_margin_15pct',
        'gate_type': 'training_margin',
        'threshold': 'official warm-start candidate target MaxDD <= 15% during checkpoint scoring when available',
        'implementation_target': 'checkpoint_objective_loss',
        'reason': 'prior state-conditioned BC candidates around 16-31% MaxDD do not leave enough retention margin',
    },
    {
        'priority': 7,
        'term': 'state_conditioned_risk_representation_before_reentry_floor',
        'gate_type': 'mechanism_requirement',
        'threshold': 'do not raise reentry/trend floors unless risk-state coverage and drawdown-interval behavior improve',
        'implementation_target': 'feature_state_or_auxiliary_head',
        'reason': 'state-conditioned multi-objective BC raised participation but failed wf_2022 drawdown interval',
    },
    {
        'priority': 8,
        'term': 'trend_ytd_participation_floor_with_drawdown_guard',
        'gate_type': 'joint_objective',
        'threshold': 'participation lift only active under low-vol/high-trend states and only if WF drawdown gate remains satisfied',
        'implementation_target': 'reward_or_bc_target_generation',
        'reason': 'trend-only guards and low-DD references miss YTD/current-best alignment',
    },
    {
        'priority': 9,
        'term': 'ppo_drawdown_drift_stop',
        'gate_type': 'stop_rule',
        'threshold': 'reject PPO/checkpoint if MaxDD worsens >2pp versus BC/reference or any WF MaxDD >=20%',
        'implementation_target': 'selection_history_stop_rules',
        'reason': 'prior PPO/BC screens show policy improvement can destroy risk constraints',
    },
]
recipe = pd.DataFrame(recipe_rows)

stop_rules = pd.DataFrame([
    {'rule': 'missing_official_replay_evidence', 'action': 'reject', 'details': 'candidate must have official replay or dashboard replay YTD plus all required WF windows'},
    {'rule': 'wf_maxdd_ge_20pct', 'action': 'reject', 'details': 'retained model cannot breach requested walk-forward drawdown gate'},
    {'rule': 'ytd_below_current_best_alignment', 'action': 'reject', 'details': f"YTD must be >= {TARGETS['required_current_best_alignment_ytd_pct']:.6f}%"},
    {'rule': 'ytd_below_soxlbacktest_reference', 'action': 'reject_or_downrank_reference', 'details': f"YTD below {TARGETS['soxlbacktest_reference_ytd_return_pct']:.6f}% cannot beat SOXLBacktest reference floor"},
    {'rule': 'ordinary_vs_official_gate_flip', 'action': 'reject_training_handoff', 'details': 'do not use ordinary split metrics to justify PPO/retention'},
    {'rule': 'risk_state_reentry_drift', 'action': 'stop_floor_sweep', 'details': 'if reentry/trend floor improves return but worsens drawdown, revise state representation before longer training'},
])

recipe


,priority,term,gate_type,threshold,implementation_target,reason
0,1,official_replay_metric_source_required,data_contract,all candidate metrics must come from official ...,checkpoint_selector_and_replay_pipeline,ordinary/cold split evidence has repeatedly ga...
1,2,complete_wf_coverage_gate,hard_reject,"wf_2020,wf_2022,wf_2023_2024",candidate_manifest,retention aggregate shows missing WF evidence ...
2,3,walkforward_drawdown_hard_gate_20pct,hard_reject,worst WF MaxDD < 20.00%,official_replay_checkpoint_screen,requested retained model must pass walk-forwar...
3,4,ytd_current_best_alignment_floor,hard_reject,YTD return >= 213.143462% through safe_backtes...,dashboard_replay_ytd_screen,low-drawdown paths underparticipate; current o...
4,5,soxlbacktest_buysell_reference_floor,reference_floor,YTD return >= 132.077978%,selection_score_reference_component,SOXLBacktest qlib buy/sell is a baseline floor...
5,6,warm_start_drawdown_margin_15pct,training_margin,official warm-start candidate target MaxDD <= ...,checkpoint_objective_loss,prior state-conditioned BC candidates around 1...
6,7,state_conditioned_risk_representation_before_r...,mechanism_requirement,do not raise reentry/trend floors unless risk-...,feature_state_or_auxiliary_head,state-conditioned multi-objective BC raised pa...
7,8,trend_ytd_participation_floor_with_drawdown_guard,joint_objective,participation lift only active under low-vol/h...,reward_or_bc_target_generation,trend-only guards and low-DD references miss Y...
8,9,ppo_drawdown_drift_stop,stop_rule,reject PPO/checkpoint if MaxDD worsens >2pp ve...,selection_history_stop_rules,prior PPO/BC screens show policy improvement c...


In [5]:

# Persist sidecars and compact machine-readable selection contract.
created_at = datetime.now(timezone.utc).isoformat()
contract = {
    'experiment': 'retention_ytd_training_objective_handoff_v1',
    'created_at': created_at,
    'decision': 'retention_ytd_training_objective_contract_ready_no_promotion',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_next': True,
    'next_mechanism': 'train_state_conditioned_multi_objective_ytd_gate_v1',
    'next_profile_candidate': 'state_conditioned_multi_objective_ytd_gate_v1',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'reference_baseline': 'soxlbacktest_policy_reimplementation_v1',
    'direct_comparison_baseline': [
        'retention_gap_aggregate_v1',
        'state_conditioned_multi_objective_bc_screen_v1',
        'sequence_trend_joint_warm_start_official_eval_v1',
        'soxlbacktest_qlib_walkforward_baseline_v1',
    ],
    'target_thresholds': TARGETS,
    'selected_diagnostic_candidate': closest_retention.get('candidate_id'),
    'best_low_drawdown_reference': best_low_dd.get('candidate_id') if best_low_dd else None,
    'soxlbacktest_qlib_reference_candidate': best_soxlbacktest.get('candidate') if best_soxlbacktest else None,
    'primary_failure_modes': [
        'joint_drawdown_and_ytd_return_gap',
        'low_drawdown_underparticipation',
        'official_replay_drawdown_drift_after_reentry_lift',
        'missing_or_incomplete_wf_evidence',
    ],
    'required_next_artifacts': [
        'profile.yaml registered in runtime/harness/card/registry/profile.yaml before a retained actor is kept',
        'official replay metrics for wf_2020, wf_2022, wf_2023_2024',
        'dashboard/YTD replay through safe_backtest_end',
        'checkpoint selector table with retention_ytd_loss and wf_drawdown_loss',
        'drawdown interval audit for any selected checkpoint',
    ],
    'sidecars': {
        'source_audit': rel(OUT / 'source_audit.csv'),
        'candidate_gap_table': rel(OUT / 'candidate_gap_table.csv'),
        'training_objective_recipe': rel(OUT / 'training_objective_recipe.csv'),
        'stop_rules': rel(OUT / 'stop_rules.csv'),
        'contract': rel(OUT / 'training_contract.json'),
        'selection': rel(OUT / 'selection.json'),
        'summary': rel(OUT / 'summary.json'),
        'conclusion': rel(OUT / 'conclusion.md'),
    },
}

summary = {
    'experiment': contract['experiment'],
    'created_at': created_at,
    'decision': contract['decision'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_next': True,
    'next_mechanism': contract['next_mechanism'],
    'next_profile_candidate': contract['next_profile_candidate'],
    'safe_backtest_end': TARGETS['safe_backtest_end'],
    'target_thresholds': TARGETS,
    'closest_retention_candidate': closest_retention.get('candidate_id'),
    'closest_retention_objective_loss': closest_retention.get('objective_loss'),
    'best_low_drawdown_reference': best_low_dd.get('candidate_id') if best_low_dd else None,
    'best_low_drawdown_ytd_return_pct': best_low_dd.get('ytd_return_pct') if best_low_dd else None,
    'soxlbacktest_reference_ytd_return_pct': TARGETS['soxlbacktest_reference_ytd_return_pct'],
    'required_current_best_alignment_ytd_pct': TARGETS['required_current_best_alignment_ytd_pct'],
    'recipe_terms': recipe['term'].tolist(),
    'stop_rule_count': int(len(stop_rules)),
    'sidecars': contract['sidecars'],
}

source_audit_df.to_csv(OUT / 'source_audit.csv', index=False)
gap_table.to_csv(OUT / 'candidate_gap_table.csv', index=False)
recipe.to_csv(OUT / 'training_objective_recipe.csv', index=False)
stop_rules.to_csv(OUT / 'stop_rules.csv', index=False)
(OUT / 'training_contract.json').write_text(json.dumps(clean_json(contract), indent=2, allow_nan=False), encoding='utf-8')
(OUT / 'selection.json').write_text(json.dumps(clean_json(contract), indent=2, allow_nan=False), encoding='utf-8')
(OUT / 'summary.json').write_text(json.dumps(clean_json(summary), indent=2, allow_nan=False), encoding='utf-8')
summary


{'experiment': 'retention_ytd_training_objective_handoff_v1',
 'created_at': '2026-06-13T14:06:52.481747+00:00',
 'decision': 'retention_ytd_training_objective_contract_ready_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_next': True,
 'next_mechanism': 'train_state_conditioned_multi_objective_ytd_gate_v1',
 'next_profile_candidate': 'state_conditioned_multi_objective_ytd_gate_v1',
 'safe_backtest_end': '2026-06-11',
 'target_thresholds': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'wf_max_drawdown_hard_gate_pct': 20.0,
  'soxlbacktest_reference_ytd_return_pct': 132.077978,
  'current_best_ytd_return_pct': 224.361539,
  'required_current_best_alignment_ytd_pct': 213.143462,
  'official_internal_relative_return_floor_pp': 30.0,
  'official_internal_maxdd_margin_pct': 15.0,
  'safe_backtest_end': '2026-06-11',
  'latest_data_date': '2026-06-12'},
 'closest_retention_candidate': 'existing_model_retention_gate::

In [6]:

lines = [
    '# Retention YTD Training Objective Handoff v1 Conclusion',
    '',
    f"Decision: `{summary['decision']}`.",
    '',
    '## Current Evidence',
    '',
    f"- Closest retained-model candidate remains `{summary['closest_retention_candidate']}`; it is diagnostic only and not promotable.",
    f"- Best low-drawdown reference remains `{summary['best_low_drawdown_reference']}` with YTD `{summary['best_low_drawdown_ytd_return_pct']:.2f}%`, far below the required current-best alignment floor `{summary['required_current_best_alignment_ytd_pct']:.2f}%`.",
    f"- SOXLBacktest qlib buy/sell reference floor is `{summary['soxlbacktest_reference_ytd_return_pct']:.2f}%`; it is a reference baseline, not runnable inference.",
    f"- Current dashboard replay source reports safe_backtest_end `{summary['safe_backtest_end']}`.",
    '',
    '## Training Contract',
    '',
    '- Next branch/profile candidate: `state_conditioned_multi_objective_ytd_gate_v1`.',
    '- Candidate screening must use official replay metrics for all WF windows plus dashboard/YTD replay through safe_backtest_end.',
    '- Hard reject any checkpoint with missing WF evidence, worst WF MaxDD >=20%, or YTD return below 213.143462%.',
    '- Keep the stricter <=15% MaxDD margin as a training/checkpoint objective, because prior warm-start candidates near 16-31% MaxDD do not leave enough retention margin.',
    '- Do not increase reentry/trend participation floors unless risk-state coverage and drawdown interval behavior improve together.',
    '',
    '## Stop Rules',
    '',
]
for row in stop_rules.to_dict('records'):
    lines.append(f"- `{row['rule']}` -> {row['action']}: {row['details']}")
lines.extend([
    '',
    '## Sidecars',
    '',
    '- `candidate_gap_table.csv`',
    '- `training_objective_recipe.csv`',
    '- `stop_rules.csv`',
    '- `training_contract.json` / `selection.json` / `summary.json`',
])
conclusion = '\n'.join(lines).rstrip() + '\n'
(OUT / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


# Retention YTD Training Objective Handoff v1 Conclusion

Decision: `retention_ytd_training_objective_contract_ready_no_promotion`.

## Current Evidence

- Closest retained-model candidate remains `existing_model_retention_gate::mode_conditioned_rebound_reward_train_v2`; it is diagnostic only and not promotable.
- Best low-drawdown reference remains `soxlbacktest_qlib_reference::wf_pass_ytd18_15_15_02` with YTD `18.31%`, far below the required current-best alignment floor `213.14%`.
- SOXLBacktest qlib buy/sell reference floor is `132.08%`; it is a reference baseline, not runnable inference.
- Current dashboard replay source reports safe_backtest_end `2026-06-11`.

## Training Contract

- Next branch/profile candidate: `state_conditioned_multi_objective_ytd_gate_v1`.
- Candidate screening must use official replay metrics for all WF windows plus dashboard/YTD replay through safe_backtest_end.
- Hard reject any checkpoint with missing WF evidence, worst WF MaxDD >=20%, or YTD return belo